In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

cat = df.select_dtypes(include=['object', 'str']).columns.tolist()
num = df.select_dtypes(exclude='object').columns.drop('converted').tolist()

print(df.isnull().sum())
df[cat] = df[cat].fillna('NA')
df[num] = df[num].fillna(0.0)

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [3]:
df['industry'].mode()[0]

'technology'

In [4]:
corr = df[num].corr()
pairs = [('interaction_count', 'lead_score'),
         ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'),
         ('annual_income', 'interaction_count')]
for a, b in pairs:
    print(a,b,corr.loc[a, b])

interaction_count lead_score 0.9157457980418697
number_of_courses_viewed lead_score 0.7572042392104886
number_of_courses_viewed interaction_count 0.7216090038937035
annual_income interaction_count 0.12284235135955959


In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values
df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])

In [6]:
for c in ['industry', 'location', 'lead_source', 'employment_status']:
    print(c, round(mutual_info_score(df_train[c], y_train),2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


In [7]:
features = cat+num
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
base_acc = accuracy_score(y_val, model.predict(X_val))
print(round(base_acc, 2))

0.65


In [10]:
def acc_with(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    Xt = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    Xv = dv.transform(df_val[feats].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(Xt, y_train)
    return accuracy_score(y_val, m.predict(Xv))

for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    a = acc_with([x for x in features if x != f])
    print(f, base_acc - a)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


In [11]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(acc_with(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
